# 04 Effective-number reweighting

Same as the baseline, but the loss of every training image is weighted by its FST group, using the effective number of samples of Cui et al. (2019) with beta = 0.999. The smallest group (FST V-VI) gets the largest weight.

The training itself is in `training.py` and is the same for all conditions. Each seed takes about 30 to 50 minutes on a laptop GPU. The results are written to `seed_<n>/` in this folder.

In [ ]:
import sys
from pathlib import Path

# The shared code is in the repository root. Look for it from the folder of this notebook upwards.
REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "analysis.py").is_file())
sys.path.insert(0, str(REPO_DIR))

import analysis as A
import training as T

CONDITION = "reweighting_effective_number"
SEEDS = [42, 43, 44]

print(CONDITION, "-", A.CONDITIONS[CONDITION]["description"])
print("Device:", T.DEVICE)
print("Results go to:", A.condition_dir(CONDITION))

In [ ]:
# The same splits for every condition. load_splits() checks the file hashes and the image files.
train_df, val_df, test_df, split_hashes = A.load_splits()
print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})
print(A.pd.crosstab(train_df.fst_group, train_df.binary_label).rename(columns={0: "benign", 1: "malignant"}))

In [ ]:
# Loss weights of this condition, computed from the training split.
# The same table is saved in every seed folder as training_weight_table.csv.
sample_weight, weight_table = A.make_sample_weights(train_df, A.CONDITIONS[CONDITION]["weighting"])
print(f"mean weight = {sample_weight.mean():.4f}, min = {sample_weight.min():.4f}, max = {sample_weight.max():.4f}")
weight_table

In [ ]:
# Train all seeds. A seed that already finished is skipped, so this cell can be started
# again after an interruption. To redo a seed: T.run_condition(CONDITION, 42, overwrite=True)
for seed in SEEDS:
    T.run_condition(CONDITION, seed)

In [ ]:
# Test results per seed, and the mean (SD) over the seeds.
per_seed = T.summarise_condition(CONDITION)